# Семинар 09. HTTP и JSON


## Цели

После семинара вы сможете:

- разбирать URL, HTTP-запрос и HTTP-ответ;
- выбирать подходящий HTTP-метод;
- сериализовать JSON и безопасно обрабатывать сетевые ответы.

> **Формат:** справочный материал для индивидуального проекта. Отдельного задания по семинару нет.

## Перед началом

Установите requests. Для сетевых примеров требуется подключение к интернету.


Python позволяет создавать веб-приложения и взаимодействовать с внешними сервисами.

### Протокол HTTP
HTTP — основной прикладной протокол веба. Рассмотрим его на примере запроса к поисковому сервису.

В браузере при этом набран запрос `https://ya.ru/search/?text=python`. Он состоит из следующих частей
  * https:// - это протокол (безопасный http)
  * ya.ru - это сервер, к которому идет обращение
  * search - путь
  * ? разделитель между путем и параметрами запроса
  * text=python - параметры запроса

При загрузке страницы браузер отправляет запрос, который включает:
  * заголовков, например `User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/83.0.4103.116 Safari/537.36`, чтобы дать знать сервису, кто именно в него пришел. Это важно и для статистики и для того, чтобы принять решение, настольную/мобильную версию сайта показать, или сделать что-то еще. Авторизационная информация в виде различных токенов или кук тоже передается в заголовках (об этом потом)
  * Метода (в данном случае GET, но бывают и другие, например POST/PUT/DELETE)
  * Пути и параметров

Браузер загружает страницу, выполняет скрипты, применяет стили и показывает результат пользователю. Для взаимодействия программ удобнее получать данные в машиночитаемом формате. В Python для HTTP-запросов есть встроенный модуль `urllib` и сторонняя библиотека `requests`, установленная в окружении курса.


In [ ]:
import requests

response = None
try:
    response = requests.get("https://example.com", timeout=5)
    response.raise_for_status()
    print(response)
except requests.RequestException as error:
    print(f"Request failed: {error}")


Что такое `<Response [200]>` и где результаты поиска?
HTTP использует стандартные трёхзначные коды состояния. Они делятся на пять семейств: 1xx, 2xx, 3xx, 4xx и 5xx.
  * 1xx - информационные ответы
  * 2xx - успешные ответы. Все ОК. Сервис успешно принял и обработал запрос
  * 3xx - перенаправления. Сервер знает, что по запрошенному адресу ничего нет, и знает, куда именно нужно перенаправить обратившегося с его запросом, чтобы он был выполнен там.
  * 4xx — ошибки клиента: некорректный запрос, отсутствие доступа, превышение квоты и т. д.
  * 5xx - ошибки сервера. Код обработки запроса завершился с неожиданной ошибкой, либо обработчик в принципе отсутствует и незапущен, 
*Все это интересно, а результаты-то где?*

Результаты доступны в общем случае при обращении к свойству `text`

In [ ]:
if response is not None:
    print(response.text[:500])  # выводим только начало ответа


HTML удобно отображать в браузере, но программе проще работать со структурированными данными. Поэтому рассмотрим машиночитаемый формат на более простом примере.

### Формат JSON
JSON (JavaScript Object Notation) — распространённый текстовый формат для передачи структурированных данных. Его значения определяются рекурсивно:
  * Дробные числа записываются как обычные числа с разделителем  `.`. Например, `1, 2.0, -100500` - все это валидные JSON представления чисел
  * Строки записываются в двойных кавычках. `"this is a string", "это строка", ""`
  * логические значения `true` и `false` так и записываются
  * массивы записываются в квадратных скобках `[1, 2, 3], ["1", [], [[]]]`. - валидные массивы *где-то такую запись мы уже видели* 
  * словари записываются в фигурных скобках `{"key": "value", "key2": []}` *тоже что-то знакомое*

Для работы с JSON в Python есть встроенный модуль `json`.

In [ ]:
import json
data = {
    "name": "John",
    "age": 30,
    "city": "New York"
}

dumped = json.dumps(data)
print(dumped)

unpacked = json.loads(dumped)
print(unpacked)

In [ ]:
import requests

# Метод json() десериализует тело ответа в объект Python.
try:
    response = requests.get("https://v2.jokeapi.dev/joke/Programming", timeout=5)
    response.raise_for_status()
    result = response.json()

    if "setup" in result:
        print("setup:", result["setup"])
        print("delivery:", result["delivery"])
    else:
        print("joke:", result["joke"])
except (requests.RequestException, ValueError) as error:
    print(f"Could not load JSON: {error}")


## HTTP методы и модифицирующие действия
В прошлом примере мы получали данные, которые нам отдавал сервер по какому-то запросу. Сами данные на сервере при этом никак не менялись. Но что, если их надо менять? Как то же туда попадают новые шутки, меняются существующие и удаляются несмешные/неприемлемые. Для этого в протоколе HTTP предусмотрены специальные методы. Прежде, чем перейти к их описанию, рассмотрим следующие свойства:
  * Кэшируемость. На всех уровнях от браузера и промежуточных серверов, может так случиться, что запомнить ранее отданный результат по какому-то запросу и отдать его же при получении такого же запроса, а не выполнять сам запрос еще раз, сильно быстрее и экономичнее (и иногда так и делают при запросе так называемых статических файлов: картинок, таблиц стилей и тд)
  * Идемпотентность. Метод `идемпотентен`, если два и более его вызова с одними и теми же параметрами приводят к тому же самому результату, как если бы он был вызван ровно один раз.
  * Повторные вызовы. Из-за сетевых проблем и всякого рода эвристик программа может принять решение выполнить один и тот же запрос несколько раз без каких-либо действий со стороны пользователя.

  **Метод GET**
  Предназначен для получения данных. Идемпотентен, повторные вызовы не приводят к изменению состояния сервера.
  
  **Нельзя через метод GET реализовывать модифицирующие действия**
  Вам никто это не запретит явно, но если вы так сделаете, можете столкнуться с неприятными последствиями: действия *иногда* будут совершаться, когда не нужно (из-за непрозрачных повторных запросов) и *время от времени* не будут совершаться, когда нужно, из-за кэширования.

  **Метод POST**
  Предназначен для отправки данных на сервер. В общем случае не идемпотентен. Чаще всего используется для создания новых сущностей: регистрации пользователя, записи в блоге или покупки. Из-за сетевых сбоев запрос может быть отправлен повторно, поэтому для критичных операций применяют ключи идемпотентности.

  **Метод PUT**
  Предназначен для создания или полной замены представления ресурса. Метод идемпотентен: повтор одинакового запроса должен приводить к тому же состоянию ресурса.

  **Метод DELETE**
  Предназначен для удаления сущности на сервере. Должен быть идемпотентен. Сервер может вернуть как пустой ответ, так и тело с результатом операции.

  **Метод PATCH**
  Предназначен для частичного обновления ресурса. PATCH не обязан быть идемпотентным: это зависит от семантики конкретной операции.

Есть и другие, но эти основные. Для каждого из них в библиотеке `requests` есть соответствующий метод.


### Дополнительный источник
https://mixedanalytics.com/blog/list-actually-free-open-no-auth-needed-apis/